# Validate JWTs Like A Library Should, And See Why Each Check Exists

Companion notebook for the [Sessions, Tokens and JWT Pitfalls](https://codemia.io/courses/security_engineering/sec_sessions_tokens_and_jwt_pitfalls) lesson on Codemia.

**What you will build**: a lab that takes JSON Web Tokens apart with PyJWT. You will decode a token by hand to see that the
payload is only encoded, not encrypted, then build a validator one check at a time and feed it the
tokens each check exists to stop: an unsigned token, a token signed for another service, an expired
token, and the classic algorithm confusion where a public key is used as an HMAC secret. Finally you
will measure how a short lifetime and a revocation list trade off against each other.

## Setup

In [ ]:
%pip install -q pyjwt cryptography numpy matplotlib

In [ ]:
import base64
import hashlib
import hmac
import json
import time

import jwt
import numpy as np
import matplotlib.pyplot as plt
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import ed25519

NOW = int(time.time())       # PyJWT checks exp and nbf against the real clock, so issue relative to it
ISSUER = 'https://auth.example.com'
AUDIENCE = 'billing-api'

signing_key = ed25519.Ed25519PrivateKey.generate()
public_key = signing_key.public_key()
public_pem = public_key.public_bytes(serialization.Encoding.PEM, serialization.PublicFormat.SubjectPublicKeyInfo)

def issue(claims, key=signing_key, alg='EdDSA', kid='2026-11'):
    base = {'iss': ISSUER, 'aud': AUDIENCE, 'sub': 'user_17', 'iat': NOW - 60, 'exp': NOW + 600}
    base.update(claims)
    base = {k: v for k, v in base.items() if v is not None}
    return jwt.encode(base, key, algorithm=alg, headers={'kid': kid})

print('PyJWT', jwt.__version__)

## Concept recap

A JWT is three base64url pieces joined by dots: a header naming the algorithm and key, a JSON payload
of claims, and a signature over the first two. The signature proves who issued the token and that it
has not changed. It does not hide the claims, and it says nothing about whether this service should
accept the token: that is what the claim checks do. Most JWT vulnerabilities are a validator that
trusts the header to choose how to verify, or that skips one of the claim checks.

## Decode a token by hand

In [ ]:
token = issue({'role': 'admin', 'email_verified': True})
header_b64, payload_b64, sig_b64 = token.split('.')

def b64url_decode(part):
    return base64.urlsafe_b64decode(part + '=' * (-len(part) % 4))

print('header :', json.loads(b64url_decode(header_b64)))
print('payload:', json.loads(b64url_decode(payload_b64)))
print('signature bytes:', len(b64url_decode(sig_b64)))

No key was needed. Anything in the payload is readable by whoever holds the token, including browser
extensions, proxies and log pipelines that record Authorization headers. Keep secrets and personal
data out of claims, or use an encrypted token format, and never log raw tokens.

## A validator, one check at a time

In [ ]:
ALLOWED_ALGS = ['EdDSA']
KEYS = {'2026-11': public_key}

def validate(token, leeway=30, max_age=3600):
    try:
        header = jwt.get_unverified_header(token)
        alg, kid = header.get('alg'), header.get('kid')
        if alg not in ALLOWED_ALGS:
            return f'reject: alg {alg!r} not allowed'
        if kid not in KEYS:
            return f'reject: unknown key {kid!r}'
        claims = jwt.decode(
            token, KEYS[kid], algorithms=ALLOWED_ALGS,     # the allow list, never the header
            audience=AUDIENCE, issuer=ISSUER, leeway=leeway,
            options={'require': ['exp', 'iat', 'iss', 'aud', 'sub']},
        )
        if NOW - claims['iat'] > max_age + leeway:
            return 'reject: token too old'
        return f"accept: sub={claims['sub']}"
    except jwt.InvalidTokenError as e:
        return f'reject: {type(e).__name__}: {e}'

print(validate(issue({})))

The structure mirrors the order the CodeExecutor in this lesson asks for. The algorithm is checked
against a list the server chose before any cryptography runs, and the key comes from the server's own
table indexed by `kid`, never from anything else in the header. Only then is the signature verified,
and only after the signature passes are the claims trusted enough to check.

## Tokens each check exists to stop

In [ ]:
def unsigned(claims):
    h = base64.urlsafe_b64encode(json.dumps({'alg': 'none', 'typ': 'JWT', 'kid': '2026-11'}).encode()).rstrip(b'=')
    base = {'iss': ISSUER, 'aud': AUDIENCE, 'sub': 'user_17', 'iat': NOW, 'exp': NOW + 600, **claims}
    p = base64.urlsafe_b64encode(json.dumps(base).encode()).rstrip(b'=')
    return (h + b'.' + p + b'.').decode()

def hs256_with(secret, claims):
    # build the token by hand: PyJWT itself refuses to use a PEM public key as an HMAC secret
    h = base64.urlsafe_b64encode(json.dumps({'alg': 'HS256', 'typ': 'JWT', 'kid': '2026-11'}).encode()).rstrip(b'=')
    base = {'iss': ISSUER, 'aud': AUDIENCE, 'sub': 'user_1', 'iat': NOW, 'exp': NOW + 600, **claims}
    p = base64.urlsafe_b64encode(json.dumps(base).encode()).rstrip(b'=')
    s = base64.urlsafe_b64encode(hmac.new(secret, h + b'.' + p, hashlib.sha256).digest()).rstrip(b'=')
    return (h + b'.' + p + b'.' + s).decode()

other_issuer_key = ed25519.Ed25519PrivateKey.generate()
cases = {
    'well formed':                     issue({}),
    'alg none, no signature':          unsigned({'role': 'admin'}),
    'HS256 keyed with our public key': hs256_with(public_pem, {'role': 'admin'}),
    'payload edited after signing':    None,
    'signed by a key we do not trust': issue({}, key=other_issuer_key),
    'unknown kid':                     issue({}, kid='attacker-key'),
    'minted for another service':      issue({'aud': 'reporting-api'}),
    'from another issuer':             issue({'iss': 'https://auth.evil.example'}),
    'expired 5 minutes ago':           issue({'exp': NOW - 300}),
    'expired 10 seconds ago':          issue({'exp': NOW - 10}),
    'not valid until tomorrow':        issue({'nbf': NOW + 86_400}),
    'no expiry at all':                issue({'exp': None}),
    'issued two days ago, long exp':   issue({'iat': NOW - 172_800, 'exp': NOW + 86_400 * 30}),
}
h, p, s = issue({}).split('.')
edited = json.loads(b64url_decode(p)); edited['sub'] = 'user_1'
cases['payload edited after signing'] = '.'.join([h, base64.urlsafe_b64encode(json.dumps(edited).encode()).rstrip(b'=').decode(), s])

for name, tok in cases.items():
    print(f'{name:<34} {validate(tok)}')

Every row but the first is rejected, and each for its own reason. Two deserve a second look.

The HS256 token is algorithm confusion. The server's verification key is public by design, so an
attacker can compute an HMAC with it as the secret. A validator that takes `alg` from the header and
passes the same key object to whichever algorithm it names would accept that token. Pinning the
algorithm list, and keeping a separate key type per algorithm, closes it; modern PyJWT also refuses to
use a PEM public key as an HMAC secret, which is why the token here had to be built by hand.

The token that expired ten seconds ago is accepted only because of the 30 second leeway, which is
there for clock skew between servers. Leeway should be seconds, not minutes, since every second of it
extends the life of every stolen token.

## Revocation versus lifetime

In [ ]:
# A JWT cannot be recalled once issued. Model a stolen token: how long is it usable, and what does
# it cost to make that shorter? A deny list checked on every request ends it within the list's
# propagation delay; otherwise it lives until exp.
rng = np.random.default_rng(5)
lifetimes = np.array([5, 15, 60, 240, 1440])            # minutes
stolen_at = rng.uniform(0, 1, 10_000)                   # fraction of the lifetime already used
for propagation in (None, 1):
    label = 'no deny list' if propagation is None else f'deny list, {propagation} min to propagate'
    print(label)
    for L in lifetimes:
        remaining = (1 - stolen_at) * L
        exposure = remaining if propagation is None else np.minimum(remaining, propagation + 5)  # 5 min to detect
        print(f'  lifetime {L:>5} min: median usable after theft {np.median(exposure):>7.1f} min, '
              f'p95 {np.percentile(exposure, 95):>7.1f} min')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
for propagation, style in ((None, '-'), (1, '--')):
    p95 = []
    for L in lifetimes:
        remaining = (1 - stolen_at) * L
        exposure = remaining if propagation is None else np.minimum(remaining, propagation + 5)
        p95.append(np.percentile(exposure, 95))
    ax.loglog(lifetimes, p95, style, marker='o', label='no deny list' if propagation is None else 'deny list')
ax.set_xlabel('access token lifetime, minutes')
ax.set_ylabel('p95 minutes a stolen token works')
ax.legend()
ax.set_title('Short lifetimes bound theft; a deny list caps it')
plt.tight_layout()
plt.show()

Without revocation, a stolen token works for as long as it has left, so a day long access token is a
day long breach. Short lifetimes, five to fifteen minutes, with a refresh token that is checked
against server state on every use, bound the damage without a lookup on every request. A deny list
caps exposure at detection plus propagation regardless of lifetime, at the price of a shared lookup,
which is exactly the session store JWTs were supposed to remove. Most systems combine short access
tokens with revocable refresh tokens and keep a small deny list for emergencies.

## Exercises

1. Write the naive validator the HS256 row defeats: read `alg` from the header and verify with
   whichever algorithm it names using the PEM bytes as the key. Confirm it accepts the forged admin
   token, then fix it with a single line.
2. Add a `jti` claim and a replay cache to `validate`, so the same token is accepted only once within
   its lifetime. What does the cache need to hold, and for how long?
3. Rotate keys: add a `2027-02` key to `KEYS`, issue tokens with both, then remove the old key. Which
   tokens start failing, and when is it safe to remove it?
4. Change the leeway to 600 seconds and rerun the case table. Which rows flip, and what does that
   cost in the revocation experiment?

## What to take away

- A JWT payload is public. Treat it as signed, not secret, and keep tokens out of logs.
- The server chooses the algorithm and the key. Never let the header decide how a token is verified.
- Verify the signature first, then require and check iss, aud, exp, nbf and iat. Each missing check
  is a distinct, real attack.
- Leeway absorbs clock skew in seconds. More than that extends every stolen token.
- A JWT cannot be recalled. Keep access tokens short, make refresh tokens revocable, and hold a deny
  list for emergencies.